

```
# This is formatted as code
```

# Inesh 4-21

### load baseline llm

In [1]:
# Cell 1: install / refresh core packages only
!pip install -U transformers accelerate bitsandbytes sentencepiece

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.6/10.6 MB 115.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 44.3 MB/s eta 0:00:00
  Attempting uninstall: transformers
    Found existing installation: transformers 5.0.0
    Uninstalling transformers-5.0.0:
      Successfully uninstalled transformers-5.0.0


In [2]:
# Cell 3: imports

import torch
from transformers import AutoTokenizer, AutoModelForCausalLM


# Cell 4: pick model
MODEL_NAME = "Qwen/Qwen2.5-7B-Instruct"


# Cell 5: load tokenizer
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
print("Tokenizer loaded")

# Cell 6: load model
model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    torch_dtype="auto",
    device_map="auto"
)
print("Model loaded")

# Cell 7: simple chat test
messages = [
    {"role": "system", "content": "You are a helpful assistant."},
    {"role": "user", "content": "Say hello in one sentence."}
]

text = tokenizer.apply_chat_template(
    messages,
    tokenize=False,
    add_generation_prompt=True
)

inputs = tokenizer(text, return_tensors="pt").to(model.device)

with torch.no_grad():
    outputs = model.generate(
        **inputs,
        max_new_tokens=60
    )

response = tokenizer.decode(outputs[0], skip_special_tokens=True)
print(response)





config.json:   0%|          | 0.00/663 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

...

model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 4 files:   0%|          | 0/4 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/339 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/243 [00:00<?, ?B/s]

Model loaded
system
You are a helpful assistant.
user
Say hello in one sentence.
assistant
Hello!


### test baseline on attention is all you need

In [3]:
# Cell 1: imports
import re
import math
from collections import Counter

article_text = """
Provided proper attribution is provided, Google hereby grants permission to
reproduce the tables and figures in this paper solely for use in journalistic or
scholarly works.
Attention Is All You Need
Ashish Vaswani∗
Google Brain
avaswani@google.com
Noam Shazeer∗
Google Brain
noam@google.com
Niki Parmar∗
Google Research
nikip@google.com
Jakob Uszkoreit∗
Google Research
usz@google.com
Llion Jones∗
Google Research
llion@google.com
Aidan N. Gomez∗ †
University of Toronto
aidan@cs.toronto.edu
Łukasz Kaiser∗
Google Brain
lukaszkaiser@google.com
Illia Polosukhin∗ ‡
illia.polosukhin@gmail.com
Abstract
The dominant sequence transduction models are based on complex recurrent or
convolutional neural networks that include an encoder and a decoder. The best
performing models also connect the encoder and decoder through an attention
mechanism. We propose a new simple network architecture, the Transformer,
based solely on attention mechanisms, dispensing with recurrence and convolutions
entirely. Experiments on two machine translation tasks show these models to
be superior in quality while being more parallelizable and requiring significantly
less time to train. Our model achieves 28.4 BLEU on the WMT 2014 English-
to-German translation task, improving over the existing best results, including
ensembles, by over 2 BLEU. On the WMT 2014 English-to-French translation task,
our model establishes a new single-model state-of-the-art BLEU score of 41.8 after
training for 3.5 days on eight GPUs, a small fraction of the training costs of the
best models from the literature. We show that the Transformer generalizes well to
other tasks by applying it successfully to English constituency parsing both with
large and limited training data.
∗Equal contribution. Listing order is random. Jakob proposed replacing RNNs with self-attention and started
the effort to evaluate this idea. Ashish, with Illia, designed and implemented the first Transformer models and
has been crucially involved in every aspect of this work. Noam proposed scaled dot-product attention, multi-head
attention and the parameter-free position representation and became the other person involved in nearly every
detail. Niki designed, implemented, tuned and evaluated countless model variants in our original codebase and
tensor2tensor. Llion also experimented with novel model variants, was responsible for our initial codebase, and
efficient inference and visualizations. Lukasz and Aidan spent countless long days designing various parts of and
implementing tensor2tensor, replacing our earlier codebase, greatly improving results and massively accelerating
our research.
†Work performed while at Google Brain.
‡Work performed while at Google Research.
31st Conference on Neural Information Processing Systems (NIPS 2017), Long Beach, CA, USA.
arXiv:1706.03762v7 [cs.CL] 2 Aug 2023
1 Introduction
Recurrent neural networks, long short-term memory [13] and gated recurrent [7] neural networks
in particular, have been firmly established as state of the art approaches in sequence modeling and
transduction problems such as language modeling and machine translation [35, 2, 5]. Numerous
efforts have since continued to push the boundaries of recurrent language models and encoder-decoder
architectures [38, 24, 15].
Recurrent models typically factor computation along the symbol positions of the input and output
sequences. Aligning the positions to steps in computation time, they generate a sequence of hidden
states ht, as a function of the previous hidden state ht−1 and the input for position t. This inherently
sequential nature precludes parallelization within training examples, which becomes critical at longer
sequence lengths, as memory constraints limit batching across examples. Recent work has achieved
significant improvements in computational efficiency through factorization tricks [21] and conditional
computation [32], while also improving model performance in case of the latter. The fundamental
constraint of sequential computation, however, remains.
Attention mechanisms have become an integral part of compelling sequence modeling and transduc-
tion models in various tasks, allowing modeling of dependencies without regard to their distance in
the input or output sequences [2, 19]. In all but a few cases [27], however, such attention mechanisms
are used in conjunction with a recurrent network.
In this work we propose the Transformer, a model architecture eschewing recurrence and instead
relying entirely on an attention mechanism to draw global dependencies between input and output.
The Transformer allows for significantly more parallelization and can reach a new state of the art in
translation quality after being trained for as little as twelve hours on eight P100 GPUs.
2 Background
The goal of reducing sequential computation also forms the foundation of the Extended Neural GPU
[16], ByteNet [18] and ConvS2S [9], all of which use convolutional neural networks as basic building
block, computing hidden representations in parallel for all input and output positions. In these models,
the number of operations required to relate signals from two arbitrary input or output positions grows
in the distance between positions, linearly for ConvS2S and logarithmically for ByteNet. This makes
it more difficult to learn dependencies between distant positions [12]. In the Transformer this is
reduced to a constant number of operations, albeit at the cost of reduced effective resolution due
to averaging attention-weighted positions, an effect we counteract with Multi-Head Attention as
described in section 3.2.
Self-attention, sometimes called intra-attention is an attention mechanism relating different positions
of a single sequence in order to compute a representation of the sequence. Self-attention has been
used successfully in a variety of tasks including reading comprehension, abstractive summarization,
textual entailment and learning task-independent sentence representations [4, 27, 28, 22].
End-to-end memory networks are based on a recurrent attention mechanism instead of sequence-
aligned recurrence and have been shown to perform well on simple-language question answering and
language modeling tasks [34].
To the best of our knowledge, however, the Transformer is the first transduction model relying
entirely on self-attention to compute representations of its input and output without using sequence-
aligned RNNs or convolution. In the following sections, we will describe the Transformer, motivate
self-attention and discuss its advantages over models such as [17, 18] and [9].
3 Model Architecture
Most competitive neural sequence transduction models have an encoder-decoder structure [5, 2, 35].
Here, the encoder maps an input sequence of symbol representations (x1, ..., xn) to a sequence
of continuous representations z = (z1, ..., zn). Given z, the decoder then generates an output
sequence (y1, ..., ym) of symbols one element at a time. At each step the model is auto-regressive
[10], consuming the previously generated symbols as additional input when generating the next.
2
Figure 1: The Transformer - model architecture.
The Transformer follows this overall architecture using stacked self-attention and point-wise, fully
connected layers for both the encoder and decoder, shown in the left and right halves of Figure 1,
respectively.
3.1 Encoder and Decoder Stacks
Encoder: The encoder is composed of a stack of N = 6 identical layers. Each layer has two
sub-layers. The first is a multi-head self-attention mechanism, and the second is a simple, position-
wise fully connected feed-forward network. We employ a residual connection [11] around each of
the two sub-layers, followed by layer normalization [1]. That is, the output of each sub-layer is
LayerNorm(x + Sublayer(x)), where Sublayer(x) is the function implemented by the sub-layer
itself. To facilitate these residual connections, all sub-layers in the model, as well as the embedding
layers, produce outputs of dimension dmodel = 512.
Decoder: The decoder is also composed of a stack of N = 6 identical layers. In addition to the two
sub-layers in each encoder layer, the decoder inserts a third sub-layer, which performs multi-head
attention over the output of the encoder stack. Similar to the encoder, we employ residual connections
around each of the sub-layers, followed by layer normalization. We also modify the self-attention
sub-layer in the decoder stack to prevent positions from attending to subsequent positions. This
masking, combined with fact that the output embeddings are offset by one position, ensures that the
predictions for position i can depend only on the known outputs at positions less than i.
3.2 Attention
An attention function can be described as mapping a query and a set of key-value pairs to an output,
where the query, keys, values, and output are all vectors. The output is computed as a weighted sum
3
Scaled Dot-Product Attention Multi-Head Attention
Figure 2: (left) Scaled Dot-Product Attention. (right) Multi-Head Attention consists of several
attention layers running in parallel.
of the values, where the weight assigned to each value is computed by a compatibility function of the
query with the corresponding key.
3.2.1 Scaled Dot-Product Attention
We call our particular attention "Scaled Dot-Product Attention" (Figure 2). The input consists of
queries and keys of dimension dk, and values of dimension dv . We compute the dot products of the
query with all keys, divide each by √dk, and apply a softmax function to obtain the weights on the
values.
In practice, we compute the attention function on a set of queries simultaneously, packed together
into a matrix Q. The keys and values are also packed together into matrices K and V . We compute
the matrix of outputs as:
Attention(Q, K, V ) = softmax( QKT
√dk
)V (1)
The two most commonly used attention functions are additive attention [2], and dot-product (multi-
plicative) attention. Dot-product attention is identical to our algorithm, except for the scaling factor
of 1√dk
. Additive attention computes the compatibility function using a feed-forward network with
a single hidden layer. While the two are similar in theoretical complexity, dot-product attention is
much faster and more space-efficient in practice, since it can be implemented using highly optimized
matrix multiplication code.
While for small values of dk the two mechanisms perform similarly, additive attention outperforms
dot product attention without scaling for larger values of dk [3]. We suspect that for large values of
dk, the dot products grow large in magnitude, pushing the softmax function into regions where it has
extremely small gradients 4. To counteract this effect, we scale the dot products by 1√dk
.
3.2.2 Multi-Head Attention
Instead of performing a single attention function with dmodel-dimensional keys, values and queries,
we found it beneficial to linearly project the queries, keys and values h times with different, learned
linear projections to dk, dk and dv dimensions, respectively. On each of these projected versions of
queries, keys and values we then perform the attention function in parallel, yielding dv -dimensional
4To illustrate why the dot products get large, assume that the components of q and k are independent random
variables with mean 0 and variance 1. Then their dot product, q · k = Pdk
i=1 qiki, has mean 0 and variance dk .
4
output values. These are concatenated and once again projected, resulting in the final values, as
depicted in Figure 2.
Multi-head attention allows the model to jointly attend to information from different representation
subspaces at different positions. With a single attention head, averaging inhibits this.
MultiHead(Q, K, V ) = Concat(head1, ..., headh)W O
where headi = Attention(QW Q
i , KW K
i , V W V
i )
Where the projections are parameter matrices W Q
i ∈ Rdmodel×dk , W K
i ∈ Rdmodel×dk , W V
i ∈ Rdmodel×dv
and W O ∈ Rhdv ×dmodel .
In this work we employ h = 8 parallel attention layers, or heads. For each of these we use
dk = dv = dmodel/h = 64. Due to the reduced dimension of each head, the total computational cost
is similar to that of single-head attention with full dimensionality.
3.2.3 Applications of Attention in our Model
The Transformer uses multi-head attention in three different ways:
• In "encoder-decoder attention" layers, the queries come from the previous decoder layer,
and the memory keys and values come from the output of the encoder. This allows every
position in the decoder to attend over all positions in the input sequence. This mimics the
typical encoder-decoder attention mechanisms in sequence-to-sequence models such as
[38, 2, 9].
• The encoder contains self-attention layers. In a self-attention layer all of the keys, values
and queries come from the same place, in this case, the output of the previous layer in the
encoder. Each position in the encoder can attend to all positions in the previous layer of the
encoder.
• Similarly, self-attention layers in the decoder allow each position in the decoder to attend to
all positions in the decoder up to and including that position. We need to prevent leftward
information flow in the decoder to preserve the auto-regressive property. We implement this
inside of scaled dot-product attention by masking out (setting to −∞) all values in the input
of the softmax which correspond to illegal connections. See Figure 2.
3.3 Position-wise Feed-Forward Networks
In addition to attention sub-layers, each of the layers in our encoder and decoder contains a fully
connected feed-forward network, which is applied to each position separately and identically. This
consists of two linear transformations with a ReLU activation in between.
FFN(x) = max(0, xW1 + b1)W2 + b2 (2)
While the linear transformations are the same across different positions, they use different parameters
from layer to layer. Another way of describing this is as two convolutions with kernel size 1.
The dimensionality of input and output is dmodel = 512, and the inner-layer has dimensionality
df f = 2048.
3.4 Embeddings and Softmax
Similarly to other sequence transduction models, we use learned embeddings to convert the input
tokens and output tokens to vectors of dimension dmodel. We also use the usual learned linear transfor-
mation and softmax function to convert the decoder output to predicted next-token probabilities. In
our model, we share the same weight matrix between the two embedding layers and the pre-softmax
linear transformation, similar to [30]. In the embedding layers, we multiply those weights by √dmodel.
5
Table 1: Maximum path lengths, per-layer complexity and minimum number of sequential operations
for different layer types. n is the sequence length, d is the representation dimension, k is the kernel
size of convolutions and r the size of the neighborhood in restricted self-attention.
Layer Type Complexity per Layer Sequential Maximum Path Length
Operations
Self-Attention O(n2 · d) O(1) O(1)
Recurrent O(n · d2) O(n) O(n)
Convolutional O(k · n · d2) O(1) O(logk(n))
Self-Attention (restricted) O(r · n · d) O(1) O(n/r)
3.5 Positional Encoding
Since our model contains no recurrence and no convolution, in order for the model to make use of the
order of the sequence, we must inject some information about the relative or absolute position of the
tokens in the sequence. To this end, we add "positional encodings" to the input embeddings at the
bottoms of the encoder and decoder stacks. The positional encodings have the same dimension dmodel
as the embeddings, so that the two can be summed. There are many choices of positional encodings,
learned and fixed [9].
In this work, we use sine and cosine functions of different frequencies:
P E(pos,2i) = sin(pos/100002i/dmodel )
P E(pos,2i+1) = cos(pos/100002i/dmodel )
where pos is the position and i is the dimension. That is, each dimension of the positional encoding
corresponds to a sinusoid. The wavelengths form a geometric progression from 2π to 10000 · 2π. We
chose this function because we hypothesized it would allow the model to easily learn to attend by
relative positions, since for any fixed offset k, P Epos+k can be represented as a linear function of
P Epos.
We also experimented with using learned positional embeddings [9] instead, and found that the two
versions produced nearly identical results (see Table 3 row (E)). We chose the sinusoidal version
because it may allow the model to extrapolate to sequence lengths longer than the ones encountered
during training.
4 Why Self-Attention
In this section we compare various aspects of self-attention layers to the recurrent and convolu-
tional layers commonly used for mapping one variable-length sequence of symbol representations
(x1, ..., xn) to another sequence of equal length (z1, ..., zn), with xi, zi ∈ Rd, such as a hidden
layer in a typical sequence transduction encoder or decoder. Motivating our use of self-attention we
consider three desiderata.
One is the total computational complexity per layer. Another is the amount of computation that can
be parallelized, as measured by the minimum number of sequential operations required.
The third is the path length between long-range dependencies in the network. Learning long-range
dependencies is a key challenge in many sequence transduction tasks. One key factor affecting the
ability to learn such dependencies is the length of the paths forward and backward signals have to
traverse in the network. The shorter these paths between any combination of positions in the input
and output sequences, the easier it is to learn long-range dependencies [12]. Hence we also compare
the maximum path length between any two input and output positions in networks composed of the
different layer types.
As noted in Table 1, a self-attention layer connects all positions with a constant number of sequentially
executed operations, whereas a recurrent layer requires O(n) sequential operations. In terms of
computational complexity, self-attention layers are faster than recurrent layers when the sequence
6
length n is smaller than the representation dimensionality d, which is most often the case with
sentence representations used by state-of-the-art models in machine translations, such as word-piece
[38] and byte-pair [31] representations. To improve computational performance for tasks involving
very long sequences, self-attention could be restricted to considering only a neighborhood of size r in
the input sequence centered around the respective output position. This would increase the maximum
path length to O(n/r). We plan to investigate this approach further in future work.
A single convolutional layer with kernel width k < n does not connect all pairs of input and output
positions. Doing so requires a stack of O(n/k) convolutional layers in the case of contiguous kernels,
or O(logk(n)) in the case of dilated convolutions [18], increasing the length of the longest paths
between any two positions in the network. Convolutional layers are generally more expensive than
recurrent layers, by a factor of k. Separable convolutions [6], however, decrease the complexity
considerably, to O(k · n · d + n · d2). Even with k = n, however, the complexity of a separable
convolution is equal to the combination of a self-attention layer and a point-wise feed-forward layer,
the approach we take in our model.
As side benefit, self-attention could yield more interpretable models. We inspect attention distributions
from our models and present and discuss examples in the appendix. Not only do individual attention
heads clearly learn to perform different tasks, many appear to exhibit behavior related to the syntactic
and semantic structure of the sentences.
5 Training
This section describes the training regime for our models.
5.1 Training Data and Batching
We trained on the standard WMT 2014 English-German dataset consisting of about 4.5 million
sentence pairs. Sentences were encoded using byte-pair encoding [3], which has a shared source-
target vocabulary of about 37000 tokens. For English-French, we used the significantly larger WMT
2014 English-French dataset consisting of 36M sentences and split tokens into a 32000 word-piece
vocabulary [38]. Sentence pairs were batched together by approximate sequence length. Each training
batch contained a set of sentence pairs containing approximately 25000 source tokens and 25000
target tokens.
5.2 Hardware and Schedule
We trained our models on one machine with 8 NVIDIA P100 GPUs. For our base models using
the hyperparameters described throughout the paper, each training step took about 0.4 seconds. We
trained the base models for a total of 100,000 steps or 12 hours. For our big models,(described on the
bottom line of table 3), step time was 1.0 seconds. The big models were trained for 300,000 steps
(3.5 days).
5.3 Optimizer
We used the Adam optimizer [20] with β1 = 0.9, β2 = 0.98 and ϵ = 10−9. We varied the learning
rate over the course of training, according to the formula:
lrate = d−0.5
model · min(step_num−0.5, step_num · warmup_steps−1.5) (3)
This corresponds to increasing the learning rate linearly for the first warmup_steps training steps,
and decreasing it thereafter proportionally to the inverse square root of the step number. We used
warmup_steps = 4000.
5.4 Regularization
We employ three types of regularization during training:
7
Table 2: The Transformer achieves better BLEU scores than previous state-of-the-art models on the
English-to-German and English-to-French newstest2014 tests at a fraction of the training cost.
Model BLEU Training Cost (FLOPs)
EN-DE EN-FR EN-DE EN-FR
ByteNet [18] 23.75
Deep-Att + PosUnk [39] 39.2 1.0 · 1020
GNMT + RL [38] 24.6 39.92 2.3 · 1019 1.4 · 1020
ConvS2S [9] 25.16 40.46 9.6 · 1018 1.5 · 1020
MoE [32] 26.03 40.56 2.0 · 1019 1.2 · 1020
Deep-Att + PosUnk Ensemble [39] 40.4 8.0 · 1020
GNMT + RL Ensemble [38] 26.30 41.16 1.8 · 1020 1.1 · 1021
ConvS2S Ensemble [9] 26.36 41.29 7.7 · 1019 1.2 · 1021
Transformer (base model) 27.3 38.1 3.3 · 1018
Transformer (big) 28.4 41.8 2.3 · 1019
Residual Dropout We apply dropout [33] to the output of each sub-layer, before it is added to the
sub-layer input and normalized. In addition, we apply dropout to the sums of the embeddings and the
positional encodings in both the encoder and decoder stacks. For the base model, we use a rate of
Pdrop = 0.1.
Label Smoothing During training, we employed label smoothing of value ϵls = 0.1 [36]. This
hurts perplexity, as the model learns to be more unsure, but improves accuracy and BLEU score.
6 Results
6.1 Machine Translation
On the WMT 2014 English-to-German translation task, the big transformer model (Transformer (big)
in Table 2) outperforms the best previously reported models (including ensembles) by more than 2.0
BLEU, establishing a new state-of-the-art BLEU score of 28.4. The configuration of this model is
listed in the bottom line of Table 3. Training took 3.5 days on 8 P100 GPUs. Even our base model
surpasses all previously published models and ensembles, at a fraction of the training cost of any of
the competitive models.
On the WMT 2014 English-to-French translation task, our big model achieves a BLEU score of 41.0,
outperforming all of the previously published single models, at less than 1/4 the training cost of the
previous state-of-the-art model. The Transformer (big) model trained for English-to-French used
dropout rate Pdrop = 0.1, instead of 0.3.
For the base models, we used a single model obtained by averaging the last 5 checkpoints, which
were written at 10-minute intervals. For the big models, we averaged the last 20 checkpoints. We
used beam search with a beam size of 4 and length penalty α = 0.6 [38]. These hyperparameters
were chosen after experimentation on the development set. We set the maximum output length during
inference to input length + 50, but terminate early when possible [38].
Table 2 summarizes our results and compares our translation quality and training costs to other model
architectures from the literature. We estimate the number of floating point operations used to train a
model by multiplying the training time, the number of GPUs used, and an estimate of the sustained
single-precision floating-point capacity of each GPU 5.
6.2 Model Variations
To evaluate the importance of different components of the Transformer, we varied our base model
in different ways, measuring the change in performance on English-to-German translation on the
5We used values of 2.8, 3.7, 6.0 and 9.5 TFLOPS for K80, K40, M40 and P100, respectively.
8
Table 3: Variations on the Transformer architecture. Unlisted values are identical to those of the base
model. All metrics are on the English-to-German translation development set, newstest2013. Listed
perplexities are per-wordpiece, according to our byte-pair encoding, and should not be compared to
per-word perplexities.
N dmodel dff h dk dv Pdrop ϵls train PPL BLEU params
steps (dev) (dev) ×106
base 6 512 2048 8 64 64 0.1 0.1 100K 4.92 25.8 65
(A)
1 512 512 5.29 24.9
4 128 128 5.00 25.5
16 32 32 4.91 25.8
32 16 16 5.01 25.4
(B) 16 5.16 25.1 58
32 5.01 25.4 60
(C)
2 6.11 23.7 36
4 5.19 25.3 50
8 4.88 25.5 80
256 32 32 5.75 24.5 28
1024 128 128 4.66 26.0 168
1024 5.12 25.4 53
4096 4.75 26.2 90
(D)
0.0 5.77 24.6
0.2 4.95 25.5
0.0 4.67 25.3
0.2 5.47 25.7
(E) positional embedding instead of sinusoids 4.92 25.7
big 6 1024 4096 16 0.3 300K 4.33 26.4 213
development set, newstest2013. We used beam search as described in the previous section, but no
checkpoint averaging. We present these results in Table 3.
In Table 3 rows (A), we vary the number of attention heads and the attention key and value dimensions,
keeping the amount of computation constant, as described in Section 3.2.2. While single-head
attention is 0.9 BLEU worse than the best setting, quality also drops off with too many heads.
In Table 3 rows (B), we observe that reducing the attention key size dk hurts model quality. This
suggests that determining compatibility is not easy and that a more sophisticated compatibility
function than dot product may be beneficial. We further observe in rows (C) and (D) that, as expected,
bigger models are better, and dropout is very helpful in avoiding over-fitting. In row (E) we replace our
sinusoidal positional encoding with learned positional embeddings [9], and observe nearly identical
results to the base model.
6.3 English Constituency Parsing
To evaluate if the Transformer can generalize to other tasks we performed experiments on English
constituency parsing. This task presents specific challenges: the output is subject to strong structural
constraints and is significantly longer than the input. Furthermore, RNN sequence-to-sequence
models have not been able to attain state-of-the-art results in small-data regimes [37].
We trained a 4-layer transformer with dmodel = 1024 on the Wall Street Journal (WSJ) portion of the
Penn Treebank [25], about 40K training sentences. We also trained it in a semi-supervised setting,
using the larger high-confidence and BerkleyParser corpora from with approximately 17M sentences
[37]. We used a vocabulary of 16K tokens for the WSJ only setting and a vocabulary of 32K tokens
for the semi-supervised setting.
We performed only a small number of experiments to select the dropout, both attention and residual
(section 5.4), learning rates and beam size on the Section 22 development set, all other parameters
remained unchanged from the English-to-German base translation model. During inference, we
9
Table 4: The Transformer generalizes well to English constituency parsing (Results are on Section 23
of WSJ)
Parser Training WSJ 23 F1
Vinyals & Kaiser el al. (2014) [37] WSJ only, discriminative 88.3
Petrov et al. (2006) [29] WSJ only, discriminative 90.4
Zhu et al. (2013) [40] WSJ only, discriminative 90.4
Dyer et al. (2016) [8] WSJ only, discriminative 91.7
Transformer (4 layers) WSJ only, discriminative 91.3
Zhu et al. (2013) [40] semi-supervised 91.3
Huang & Harper (2009) [14] semi-supervised 91.3
McClosky et al. (2006) [26] semi-supervised 92.1
Vinyals & Kaiser el al. (2014) [37] semi-supervised 92.1
Transformer (4 layers) semi-supervised 92.7
Luong et al. (2015) [23] multi-task 93.0
Dyer et al. (2016) [8] generative 93.3
increased the maximum output length to input length + 300. We used a beam size of 21 and α = 0.3
for both WSJ only and the semi-supervised setting.
Our results in Table 4 show that despite the lack of task-specific tuning our model performs sur-
prisingly well, yielding better results than all previously reported models with the exception of the
Recurrent Neural Network Grammar [8].
In contrast to RNN sequence-to-sequence models [37], the Transformer outperforms the Berkeley-
Parser [29] even when training only on the WSJ training set of 40K sentences.
7 Conclusion
In this work, we presented the Transformer, the first sequence transduction model based entirely on
attention, replacing the recurrent layers most commonly used in encoder-decoder architectures with
multi-headed self-attention.
For translation tasks, the Transformer can be trained significantly faster than architectures based
on recurrent or convolutional layers. On both WMT 2014 English-to-German and WMT 2014
English-to-French translation tasks, we achieve a new state of the art. In the former task our best
model outperforms even all previously reported ensembles.
We are excited about the future of attention-based models and plan to apply them to other tasks. We
plan to extend the Transformer to problems involving input and output modalities other than text and
to investigate local, restricted attention mechanisms to efficiently handle large inputs and outputs
such as images, audio and video. Making generation less sequential is another research goals of ours.
The code we used to train and evaluate our models is available at https://github.com/
tensorflow/tensor2tensor.
Acknowledgements We are grateful to Nal Kalchbrenner and Stephan Gouws for their fruitful
comments, corrections and inspiration.
References
[1] Jimmy Lei Ba, Jamie Ryan Kiros, and Geoffrey E Hinton. Layer normalization. arXiv preprint
arXiv:1607.06450, 2016.
[2] Dzmitry Bahdanau, Kyunghyun Cho, and Yoshua Bengio. Neural machine translation by jointly
learning to align and translate. CoRR, abs/1409.0473, 2014.
[3] Denny Britz, Anna Goldie, Minh-Thang Luong, and Quoc V. Le. Massive exploration of neural
machine translation architectures. CoRR, abs/1703.03906, 2017.
[4] Jianpeng Cheng, Li Dong, and Mirella Lapata. Long short-term memory-networks for machine
reading. arXiv preprint arXiv:1601.06733, 2016.
10
[5] Kyunghyun Cho, Bart van Merrienboer, Caglar Gulcehre, Fethi Bougares, Holger Schwenk,
and Yoshua Bengio. Learning phrase representations using rnn encoder-decoder for statistical
machine translation. CoRR, abs/1406.1078, 2014.
[6] Francois Chollet. Xception: Deep learning with depthwise separable convolutions. arXiv
preprint arXiv:1610.02357, 2016.
[7] Junyoung Chung, Çaglar Gülçehre, Kyunghyun Cho, and Yoshua Bengio. Empirical evaluation
of gated recurrent neural networks on sequence modeling. CoRR, abs/1412.3555, 2014.
[8] Chris Dyer, Adhiguna Kuncoro, Miguel Ballesteros, and Noah A. Smith. Recurrent neural
network grammars. In Proc. of NAACL, 2016.
[9] Jonas Gehring, Michael Auli, David Grangier, Denis Yarats, and Yann N. Dauphin. Convolu-
tional sequence to sequence learning. arXiv preprint arXiv:1705.03122v2, 2017.
[10] Alex Graves. Generating sequences with recurrent neural networks. arXiv preprint
arXiv:1308.0850, 2013.
[11] Kaiming He, Xiangyu Zhang, Shaoqing Ren, and Jian Sun. Deep residual learning for im-
age recognition. In Proceedings of the IEEE Conference on Computer Vision and Pattern
Recognition, pages 770–778, 2016.
[12] Sepp Hochreiter, Yoshua Bengio, Paolo Frasconi, and Jürgen Schmidhuber. Gradient flow in
recurrent nets: the difficulty of learning long-term dependencies, 2001.
[13] Sepp Hochreiter and Jürgen Schmidhuber. Long short-term memory. Neural computation,
9(8):1735–1780, 1997.
[14] Zhongqiang Huang and Mary Harper. Self-training PCFG grammars with latent annotations
across languages. In Proceedings of the 2009 Conference on Empirical Methods in Natural
Language Processing, pages 832–841. ACL, August 2009.
[15] Rafal Jozefowicz, Oriol Vinyals, Mike Schuster, Noam Shazeer, and Yonghui Wu. Exploring
the limits of language modeling. arXiv preprint arXiv:1602.02410, 2016.
[16] Łukasz Kaiser and Samy Bengio. Can active memory replace attention? In Advances in Neural
Information Processing Systems, (NIPS), 2016.
[17] Łukasz Kaiser and Ilya Sutskever. Neural GPUs learn algorithms. In International Conference
on Learning Representations (ICLR), 2016.
[18] Nal Kalchbrenner, Lasse Espeholt, Karen Simonyan, Aaron van den Oord, Alex Graves, and Ko-
ray Kavukcuoglu. Neural machine translation in linear time. arXiv preprint arXiv:1610.10099v2,
2017.
[19] Yoon Kim, Carl Denton, Luong Hoang, and Alexander M. Rush. Structured attention networks.
In International Conference on Learning Representations, 2017.
[20] Diederik Kingma and Jimmy Ba. Adam: A method for stochastic optimization. In ICLR, 2015.
[21] Oleksii Kuchaiev and Boris Ginsburg. Factorization tricks for LSTM networks. arXiv preprint
arXiv:1703.10722, 2017.
[22] Zhouhan Lin, Minwei Feng, Cicero Nogueira dos Santos, Mo Yu, Bing Xiang, Bowen
Zhou, and Yoshua Bengio. A structured self-attentive sentence embedding. arXiv preprint
arXiv:1703.03130, 2017.
[23] Minh-Thang Luong, Quoc V. Le, Ilya Sutskever, Oriol Vinyals, and Lukasz Kaiser. Multi-task
sequence to sequence learning. arXiv preprint arXiv:1511.06114, 2015.
[24] Minh-Thang Luong, Hieu Pham, and Christopher D Manning. Effective approaches to attention-
based neural machine translation. arXiv preprint arXiv:1508.04025, 2015.
11
[25] Mitchell P Marcus, Mary Ann Marcinkiewicz, and Beatrice Santorini. Building a large annotated
corpus of english: The penn treebank. Computational linguistics, 19(2):313–330, 1993.
[26] David McClosky, Eugene Charniak, and Mark Johnson. Effective self-training for parsing. In
Proceedings of the Human Language Technology Conference of the NAACL, Main Conference,
pages 152–159. ACL, June 2006.
[27] Ankur Parikh, Oscar Täckström, Dipanjan Das, and Jakob Uszkoreit. A decomposable attention
model. In Empirical Methods in Natural Language Processing, 2016.
[28] Romain Paulus, Caiming Xiong, and Richard Socher. A deep reinforced model for abstractive
summarization. arXiv preprint arXiv:1705.04304, 2017.
[29] Slav Petrov, Leon Barrett, Romain Thibaux, and Dan Klein. Learning accurate, compact,
and interpretable tree annotation. In Proceedings of the 21st International Conference on
Computational Linguistics and 44th Annual Meeting of the ACL, pages 433–440. ACL, July
2006.
[30] Ofir Press and Lior Wolf. Using the output embedding to improve language models. arXiv
preprint arXiv:1608.05859, 2016.
[31] Rico Sennrich, Barry Haddow, and Alexandra Birch. Neural machine translation of rare words
with subword units. arXiv preprint arXiv:1508.07909, 2015.
[32] Noam Shazeer, Azalia Mirhoseini, Krzysztof Maziarz, Andy Davis, Quoc Le, Geoffrey Hinton,
and Jeff Dean. Outrageously large neural networks: The sparsely-gated mixture-of-experts
layer. arXiv preprint arXiv:1701.06538, 2017.
[33] Nitish Srivastava, Geoffrey E Hinton, Alex Krizhevsky, Ilya Sutskever, and Ruslan Salakhutdi-
nov. Dropout: a simple way to prevent neural networks from overfitting. Journal of Machine
Learning Research, 15(1):1929–1958, 2014.
[34] Sainbayar Sukhbaatar, Arthur Szlam, Jason Weston, and Rob Fergus. End-to-end memory
networks. In C. Cortes, N. D. Lawrence, D. D. Lee, M. Sugiyama, and R. Garnett, editors,
Advances in Neural Information Processing Systems 28, pages 2440–2448. Curran Associates,
Inc., 2015.
[35] Ilya Sutskever, Oriol Vinyals, and Quoc VV Le. Sequence to sequence learning with neural
networks. In Advances in Neural Information Processing Systems, pages 3104–3112, 2014.
[36] Christian Szegedy, Vincent Vanhoucke, Sergey Ioffe, Jonathon Shlens, and Zbigniew Wojna.
Rethinking the inception architecture for computer vision. CoRR, abs/1512.00567, 2015.
[37] Vinyals & Kaiser, Koo, Petrov, Sutskever, and Hinton. Grammar as a foreign language. In
Advances in Neural Information Processing Systems, 2015.
[38] Yonghui Wu, Mike Schuster, Zhifeng Chen, Quoc V Le, Mohammad Norouzi, Wolfgang
Macherey, Maxim Krikun, Yuan Cao, Qin Gao, Klaus Macherey, et al. Google’s neural machine
translation system: Bridging the gap between human and machine translation. arXiv preprint
arXiv:1609.08144, 2016.
[39] Jie Zhou, Ying Cao, Xuguang Wang, Peng Li, and Wei Xu. Deep recurrent models with
fast-forward connections for neural machine translation. CoRR, abs/1606.04199, 2016.
[40] Muhua Zhu, Yue Zhang, Wenliang Chen, Min Zhang, and Jingbo Zhu. Fast and accurate
shift-reduce constituent parsing. In Proceedings of the 51st Annual Meeting of the ACL (Volume
1: Long Papers), pages 434–443. ACL, August 2013.
12
Attention VisualizationsInput-Input Layer5
It
is
in
this
spirit
that
a
majority
of
American
governments
have
passed
new
laws
since
2009
making
the
registration
or
voting
process
more
difficult
.
<EOS>
<pad>
<pad>
<pad>
<pad>
<pad>
<pad>
It
is
in
this
spirit
that
a
majority
of
American
governments
have
passed
new
laws
since
2009
making
the
registration
or
voting
process
more
difficult
.
<EOS>
<pad>
<pad>
<pad>
<pad>
<pad>
<pad>
Figure 3: An example of the attention mechanism following long-distance dependencies in the
encoder self-attention in layer 5 of 6. Many of the attention heads attend to a distant dependency of
the verb ‘making’, completing the phrase ‘making...more difficult’. Attentions here shown only for
the word ‘making’. Different colors represent different heads. Best viewed in color.
13
Input-Input Layer5
The
Law
will
never
be
perfect
,
but
its
application
should
be
just
-
this
is
what
we
are
missing
,
in
my
opinion
.
<EOS>
<pad>
The
Law
will
never
be
perfect
,
but
its
application
should
be
just
-
this
is
what
we
are
missing
,
in
my
opinion
.
<EOS>
<pad>Input-Input Layer5
The
Law
will
never
be
perfect
,
but
its
application
should
be
just
-
this
is
what
we
are
missing
,
in
my
opinion
.
<EOS>
<pad>
The
Law
will
never
be
perfect
,
but
its
application
should
be
just
-
this
is
what
we
are
missing
,
in
my
opinion
.
<EOS>
<pad>Figure 4: Two attention heads, also in layer 5 of 6, apparently involved in anaphora resolution. Top:
Full attentions for head 5. Bottom: Isolated attentions from just the word ‘its’ for attention heads 5
and 6. Note that the attentions are very sharp for this word.
14
Input-Input Layer5
The
Law
will
never
be
perfect
,
but
its
application
should
be
just
-
this
is
what
we
are
missing
,
in
my
opinion
.
<EOS>
<pad>
The
Law
will
never
be
perfect
,
but
its
application
should
be
just
-
this
is
what
we
are
missing
,
in
my
opinion
.
<EOS>
<pad>Input-Input Layer5
The
Law
will
never
be
perfect
,
but
its
application
should
be
just
-
this
is
what
we
are
missing
,
in
my
opinion
.
<EOS>
<pad>
The
Law
will
never
be
perfect
,
but
its
application
should
be
just
-
this
is
what
we
are
missing
,
in
my
opinion
.
<EOS>
<pad>Figure 5: Many of the attention heads exhibit behaviour that seems related to the structure of the
sentence. We give two such examples above, from two different heads from the encoder self-attention
at layer 5 of 6. The heads clearly learned to perform different tasks.
15
"""

# Cell 3: clean up article text a bit
def normalize_text(text: str) -> str:
    text = text.replace("−", "-")
    text = re.sub(r"\s+", " ", text)
    return text.strip()

article_text = normalize_text(article_text)
print(article_text[:500])
print("\nLength:", len(article_text))

# Cell 4: split article into sentences and keep character locations

def split_sentences_with_offsets(text: str):
    """
    Returns a list of dicts:
    [
      {
        "sentence": "...",
        "start_char": 123,
        "end_char": 180
      },
      ...
    ]
    """
    # Simple sentence splitter that works reasonably well for a baseline.
    pattern = re.compile(r'[^.!?]+[.!?]|[^.!?]+$')
    matches = list(pattern.finditer(text))

    sentences = []
    for m in matches:
        sent = m.group(0).strip()
        if len(sent) < 20:
            continue
        sentences.append({
            "sentence": sent,
            "start_char": m.start(),
            "end_char": m.end()
        })
    return sentences

sentences = split_sentences_with_offsets(article_text)

print("Number of sentences:", len(sentences))
print("\nFirst 5 sentences:\n")
for s in sentences[:5]:
    print(s)
    print()





# Cell 5: simple lexical similarity scorer

STOPWORDS = {
    "the","a","an","and","or","but","if","in","on","at","to","for","of","by","with",
    "is","are","was","were","be","been","being","this","that","these","those","it",
    "its","as","from","into","than","then","we","they","their","our","you","your",
    "can","could","should","would","will","may","might","do","does","did","have","has","had"
}

def tokenize(text: str):
    return re.findall(r"[a-zA-Z0-9\-]+", text.lower())

def content_words(text: str):
    return [tok for tok in tokenize(text) if tok not in STOPWORDS and len(tok) > 1]

def sentence_similarity(query: str, sentence: str) -> float:
    """
    Simple baseline score:
    - overlap of content words
    - weighted by rarity within query+sentence
    """
    q_words = content_words(query)
    s_words = content_words(sentence)

    if not q_words or not s_words:
        return 0.0

    q_counts = Counter(q_words)
    s_counts = Counter(s_words)

    vocab = set(q_counts) | set(s_counts)
    score = 0.0

    for word in vocab:
        qf = q_counts.get(word, 0)
        sf = s_counts.get(word, 0)
        if qf > 0 and sf > 0:
            # reward exact overlap
            score += 1.0 + math.log(1 + sf)

    # small bonus for substring presence of key query phrases
    q_text = " ".join(q_words)
    s_text = " ".join(s_words)
    if len(q_words) >= 2:
        for i in range(len(q_words) - 1):
            phrase = q_words[i] + " " + q_words[i+1]
            if phrase in s_text:
                score += 1.5

    # normalize a bit so long sentences do not dominate too much
    score = score / (1 + 0.05 * len(s_words))
    return score



    # Cell 6: retrieve top candidate quote sentences with locations

def get_top_candidate_quotes(query: str, sentences, top_k: int = 5):
    scored = []
    for item in sentences:
        score = sentence_similarity(query, item["sentence"])
        scored.append({
            "sentence": item["sentence"],
            "score": score,
            "start_char": item["start_char"],
            "end_char": item["end_char"]
        })

    scored = sorted(scored, key=lambda x: x["score"], reverse=True)

    # keep only nonzero scores for cleaner results
    scored = [x for x in scored if x["score"] > 0]

    return scored[:top_k]

# Example test
test_query = "What does the paper say about why the Transformer is better than recurrent models?"
top_quotes = get_top_candidate_quotes(test_query, sentences, top_k=5)

for i, q in enumerate(top_quotes, 1):
    print(f"{i}. score={q['score']:.3f}")
    print(f"   chars=({q['start_char']}, {q['end_char']})")
    print(f"   {q['sentence']}\n")


Provided proper attribution is provided, Google hereby grants permission to reproduce the tables and figures in this paper solely for use in journalistic or scholarly works. Attention Is All You Need Ashish Vaswani∗ Google Brain avaswani@google.com Noam Shazeer∗ Google Brain noam@google.com Niki Parmar∗ Google Research nikip@google.com Jakob Uszkoreit∗ Google Research usz@google.com Llion Jones∗ Google Research llion@google.com Aidan N. Gomez∗ † University of Toronto aidan@cs.toronto.edu Łukasz 

Length: 39605
Number of sentences: 392

First 5 sentences:

{'sentence': 'Provided proper attribution is provided, Google hereby grants permission to reproduce the tables and figures in this paper solely for use in journalistic or scholarly works.', 'start_char': 0, 'end_char': 173}

{'sentence': 'Attention Is All You Need Ashish Vaswani∗ Google Brain avaswani@google.', 'start_char': 173, 'end_char': 245}

{'sentence': 'com Noam Shazeer∗ Google Brain noam@google.', 'start_char': 245, 'end_char

✅ Baselines You’ve Implemented

1. Naive Sentence-Based Quote Extractor (No ML)

What it does:

* Splits article into sentences
* Scores each sentence using:
    * keyword overlap
    * simple frequency weighting
* Returns top-k sentences as “quotes”
* Tracks exact character locations ✅

Strengths:

* Fast, deterministic
* Fully interpretable (great for project writeup)
* Guarantees extracted text is real (no hallucination)

Weaknesses:

* Retrieves irrelevant sentences (e.g., references, captions, headers)
* Misses semantically relevant quotes (no understanding)
* Overweights keyword overlap → bad ranking

👉 Example failure:

* Returned:
    * “Deep recurrent models with fast-forward connections…”
    * (this is a citation/reference, not evidence)

In [4]:
# Cell 7: verify quote is exact substring of article text
# 2. Quote Verification Layer

def verify_quote_in_source(quote_text: str, full_text: str) -> bool:
    return quote_text in full_text

verified_quotes = []
for q in top_quotes:
    q["verified"] = verify_quote_in_source(q["sentence"], article_text)
    verified_quotes.append(q)

verified_quotes[:3]

[{'sentence': 'Deep recurrent models with fast-forward connections for neural machine translation.',
  'score': 3.4902102579427794,
  'start_char': 36815,
  'end_char': 36899,
  'verified': True},
 {'sentence': 'For our base models using the hyperparameters described throughout the paper, each training step took about 0.',
  'score': 3.1746509635498974,
  'start_char': 20576,
  'end_char': 20687,
  'verified': True},
 {'sentence': 'Recurrent models typically factor computation along the symbol positions of the input and output sequences.',
  'score': 3.1524479749160585,
  'start_char': 3289,
  'end_char': 3397,
  'verified': True}]

In [5]:
# Cell 8: helper to decide whether user explicitly wants quotes
# 3. Quote Tool Trigger Logic

def wants_quotes(user_query: str) -> bool:
    q = user_query.lower()
    triggers = [
        "quote", "quotes", "exact quote", "exact quotes", "verbatim",
        "direct quote", "direct quotes", "cite the sentence", "supporting quote"
    ]
    return any(t in q for t in triggers)

print(wants_quotes("Give me two exact quotes about training time"))
print(wants_quotes("Summarize the paper"))

True
False


In [6]:
# Cell 9: build evidence block for Qwen
# 4. Evidence Injection into LLM (Qwen)

def build_evidence_block(candidate_quotes, top_k: int = 3):
    chosen = candidate_quotes[:top_k]
    lines = []
    for i, q in enumerate(chosen, 1):
        lines.append(
            f"[{i}]\n"
            f"Quote: \"{q['sentence']}\"\n"
            f"Location: start_char={q['start_char']}, end_char={q['end_char']}\n"
            f"Verified: {q['verified']}\n"
            f"Score: {q['score']:.3f}"
        )
    return "\n\n".join(lines)

print(build_evidence_block(verified_quotes, top_k=2))

[1]
Quote: "Deep recurrent models with fast-forward connections for neural machine translation."
Location: start_char=36815, end_char=36899
Verified: True
Score: 3.490

[2]
Quote: "For our base models using the hyperparameters described throughout the paper, each training step took about 0."
Location: start_char=20576, end_char=20687
Verified: True
Score: 3.175


In [7]:
# Cell 10: final answer generator using Qwen

def generate_answer_with_optional_quotes(user_query: str, article_title: str, article_text: str, sentences):
    quote_mode = wants_quotes(user_query)
    candidate_quotes = get_top_candidate_quotes(user_query, sentences, top_k=5)

    for q in candidate_quotes:
        q["verified"] = verify_quote_in_source(q["sentence"], article_text)

    evidence_block = build_evidence_block(candidate_quotes, top_k=3)

    if quote_mode:
        system_prompt = (
            "You are a quote-grounded assistant.\n"
            "The user explicitly asked for quotes.\n"
            "Only use quotation marks for text that appears exactly in the provided verified quotes.\n"
            "Include the quote locations in your answer.\n"
            "Do not invent quotes.\n"
        )
        user_prompt = (
            f"Article title: {article_title}\n\n"
            f"User query: {user_query}\n\n"
            f"Verified candidate quotes:\n{evidence_block}\n\n"
            "Answer the user using the best quotes. Mention the locations."
        )
    else:
        system_prompt = (
            "You are a grounded assistant.\n"
            "The user did not explicitly ask for quotes.\n"
            "Do not use quotation marks unless absolutely necessary.\n"
            "Answer only based on the evidence provided.\n"
        )
        user_prompt = (
            f"Article title: {article_title}\n\n"
            f"User query: {user_query}\n\n"
            f"Relevant evidence:\n{evidence_block}\n\n"
            "Answer the user without direct quotes unless necessary."
        )

    messages = [
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": user_prompt},
    ]

    text = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True
    )

    inputs = tokenizer(text, return_tensors="pt").to(model.device)

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=120,
            do_sample=False
        )

    decoded = tokenizer.decode(outputs[0], skip_special_tokens=True)

    return {
        "quote_mode": quote_mode,
        "candidate_quotes": candidate_quotes,
        "raw_output": decoded
    }

In [8]:
article_title = "Attention Is All You Need"

In [9]:
# Cell 11: test a quote prompt

quote_prompt = "Give me two exact quotes about why the Transformer is better than recurrent models."
result_quote = generate_answer_with_optional_quotes(
    user_query=quote_prompt,
    article_title=article_title,
    article_text=article_text,
    sentences=sentences
)

print("QUOTE MODE:", result_quote["quote_mode"])
print("\nTOP CANDIDATES:\n")
for q in result_quote["candidate_quotes"][:3]:
    print(q)
    print()

print("\nMODEL OUTPUT:\n")
print(result_quote["raw_output"])

QUOTE MODE: True

TOP CANDIDATES:

{'sentence': 'Deep recurrent models with fast-forward connections for neural machine translation.', 'score': 3.4902102579427794, 'start_char': 36815, 'end_char': 36899, 'verified': True}

{'sentence': 'Recurrent models typically factor computation along the symbol positions of the input and output sequences.', 'score': 3.1524479749160585, 'start_char': 3289, 'end_char': 3397, 'verified': True}

{'sentence': 'We give two such examples above, from two different heads from the encoder self-attention at layer 5 of 6.', 'score': 2.4462964317600355, 'start_char': 39441, 'end_char': 39548, 'verified': True}


MODEL OUTPUT:

system
You are a quote-grounded assistant.
The user explicitly asked for quotes.
Only use quotation marks for text that appears exactly in the provided verified quotes.
Include the quote locations in your answer.
Do not invent quotes.

user
Article title: Attention Is All You Need

User query: Give me two exact quotes about why the Transf

📊 How the Model is Performing

🔴 Quote Task (Main Objective)

Result: NOT GOOD (but very informative)

What went wrong:

* Retrieval is the bottleneck (not the LLM)

The model says:

“The quotes provided do not directly compare…”

👉 That’s actually correct behavior

Interpretation:

* LLM is working properly ✅
* It refuses to hallucinate ✅
* It recognizes bad evidence ✅

BUT:

* Retriever failed to find relevant sentences ❌

In [10]:
# Cell 12: test a non-quote prompt

nonquote_prompt = "Summarize why the Transformer was important."
result_nonquote = generate_answer_with_optional_quotes(
    user_query=nonquote_prompt,
    article_title=article_title,
    article_text=article_text,
    sentences=sentences
)

print("QUOTE MODE:", result_nonquote["quote_mode"])
print("\nMODEL OUTPUT:\n")
print(result_nonquote["raw_output"])

QUOTE MODE: False

MODEL OUTPUT:

system
You are a grounded assistant.
The user did not explicitly ask for quotes.
Do not use quotation marks unless absolutely necessary.
Answer only based on the evidence provided.

user
Article title: Attention Is All You Need

User query: Summarize why the Transformer was important.

Relevant evidence:
[1]
Quote: "2 Figure 1: The Transformer - model architecture."
Location: start_char=7111, end_char=7161
Verified: True
Score: 1.411

[2]
Quote: "3 · 1018 Transformer (big) 28."
Location: start_char=22097, end_char=22127
Verified: True
Score: 1.411

[3]
Quote: "8 Table 3: Variations on the Transformer architecture."
Location: start_char=24645, end_char=24700
Verified: True
Score: 1.411

Answer the user without direct quotes unless necessary.
assistant
The Transformer was important because it introduced a novel architecture that relies solely on attention mechanisms, eliminating the need for recurrent layers. This design allowed for parallel processing o

🟡 Non-Quote Task (Secondary)

Result: DECENT despite bad evidence

Even with garbage evidence:

* Model produced a correct high-level summary

👉 Why?

* Pretrained knowledge of Transformers
* Not actually relying on your pipeline

This is important:

* Your system is NOT grounded yet
* Model is still answering from memory

In [11]:
# Cell 13: optional cleaner display of only assistant part

def extract_assistant_reply(full_decoded_text: str) -> str:
    marker = "assistant"
    idx = full_decoded_text.lower().rfind(marker)
    if idx != -1:
        return full_decoded_text[idx + len(marker):].strip()
    return full_decoded_text

print("QUOTE PROMPT FINAL ANSWER:\n")
print(extract_assistant_reply(result_quote["raw_output"]))

print("\n" + "="*80 + "\n")

print("NON-QUOTE PROMPT FINAL ANSWER:\n")
print(extract_assistant_reply(result_nonquote["raw_output"]))

QUOTE PROMPT FINAL ANSWER:

The quotes provided do not directly compare the Transformer model to recurrent models or highlight specific advantages of the Transformer. However, based on the available quotes, here are two quotes that might be relevant to the context:

1. "Recurrent models typically factor computation along the symbol positions of the input and output sequences." (Location: start_char=3289, end_char=3397)

This quote describes how recurrent models operate but does not directly state why the Transformer is better. 

For a direct comparison, we would need additional quotes that specifically address the advantages of the Transformer over recurrent models


NON-QUOTE PROMPT FINAL ANSWER:

The Transformer was important because it introduced a novel architecture that relies solely on attention mechanisms, eliminating the need for recurrent layers. This design allowed for parallel processing of input data, significantly improving training efficiency and performance in natural la

### trying to fix retrieval

In [12]:
from transformers import AutoTokenizer, AutoModel
import torch

embed_model_name = "intfloat/e5-small-v2"

embed_tokenizer = AutoTokenizer.from_pretrained(embed_model_name)
embed_model = AutoModel.from_pretrained(embed_model_name)

embed_model.eval()

config.json:   0%|          | 0.00/615 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/314 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/125 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/133M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertModel(
  (embeddings): BertEmbeddings(
    (word_embeddings): Embedding(30522, 384, padding_idx=0)
    (position_embeddings): Embedding(512, 384)
    (token_type_embeddings): Embedding(2, 384)
    (LayerNorm): LayerNorm((384,), eps=1e-12, elementwise_affine=True)
    (dropout): Dropout(p=0.1, inplace=False)
  )
  (encoder): BertEncoder(
    (layer): ModuleList(
      (0-11): 12 x BertLayer(
        (attention): BertAttention(
          (self): BertSelfAttention(
            (query): Linear(in_features=384, out_features=384, bias=True)
            (key): Linear(in_features=384, out_features=384, bias=True)
            (value): Linear(in_features=384, out_features=384, bias=True)
            (dropout): Dropout(p=0.1, inplace=False)
          )
          (output): BertSelfOutput(
            (dense): Linear(in_features=384, out_features=384, bias=True)
            (LayerNorm): LayerNorm((384,), eps=1e-12, elementwise_affine=True)
            (dropout): Dropout(p=0.1, inplace=False)
  

In [13]:
def embed_text(texts, prefix="passage"):
    """
    texts: list of strings
    prefix: "query" or "passage"
    """
    texts = [f"{prefix}: {t}" for t in texts]

    inputs = embed_tokenizer(
        texts,
        padding=True,
        truncation=True,
        return_tensors="pt"
    )

    with torch.no_grad():
        outputs = embed_model(**inputs)

    # mean pooling
    embeddings = outputs.last_hidden_state.mean(dim=1)

    # normalize (important for cosine similarity)
    embeddings = torch.nn.functional.normalize(embeddings, p=2, dim=1)

    return embeddings

In [14]:
sentence_texts = [s["sentence"] for s in sentences]

sentence_embeddings = embed_text(sentence_texts, prefix="passage")

print("Embeddings shape:", sentence_embeddings.shape)

Embeddings shape: torch.Size([392, 384])


In [15]:
def get_top_quotes_semantic(query, sentences, sentence_embeddings, top_k=5):
    # embed query
    query_emb = embed_text([query], prefix="query")  # shape (1, d)

    # cosine similarity = dot product (since normalized)
    scores = torch.matmul(sentence_embeddings, query_emb.T).squeeze()

    scored = []
    for i, item in enumerate(sentences):
        scored.append({
            "sentence": item["sentence"],
            "score": float(scores[i]),
            "start_char": item["start_char"],
            "end_char": item["end_char"]
        })

    scored = sorted(scored, key=lambda x: x["score"], reverse=True)

    return scored[:top_k]

In [16]:
test_query = "Why is the Transformer better than recurrent models?"

top_quotes_semantic = get_top_quotes_semantic(
    test_query,
    sentences,
    sentence_embeddings,
    top_k=5
)

for i, q in enumerate(top_quotes_semantic, 1):
    print(f"{i}. score={q['score']:.3f}")
    print(f"   {q['sentence']}\n")

1. score=0.880
   7 Conclusion In this work, we presented the Transformer, the first sequence transduction model based entirely on attention, replacing the recurrent layers most commonly used in encoder-decoder architectures with multi-headed self-attention.

2. score=0.867
   In contrast to RNN sequence-to-sequence models [37], the Transformer outperforms the Berkeley- Parser [29] even when training only on the WSJ training set of 40K sentences.

3. score=0.865
   In this work we propose the Transformer, a model architecture eschewing recurrence and instead relying entirely on an attention mechanism to draw global dependencies between input and output.

4. score=0.865
   Our results in Table 4 show that despite the lack of task-specific tuning our model performs sur- prisingly well, yielding better results than all previously reported models with the exception of the Recurrent Neural Network Grammar [8].

5. score=0.856
   For translation tasks, the Transformer can be trained signific

In [17]:
def is_good_sentence(sent):
    s = sent.lower()

    # remove junk
    if len(sent) < 40:
        return False
    if "table" in s or "figure" in s:
        return False
    if "@" in s:
        return False
    if "et al" in s:
        return False
    if re.match(r'^\d', s):  # starts with number like "7 Conclusion"
        return False

    return True

filtered_sentences = [s for s in sentences if is_good_sentence(s["sentence"])]



In [18]:
sentence_texts = [s["sentence"] for s in filtered_sentences]

sentence_embeddings = embed_text(sentence_texts, prefix="passage")

print("Embeddings shape:", sentence_embeddings.shape)

Embeddings shape: torch.Size([247, 384])


In [19]:
test_query = "Why is the Transformer better than recurrent models?"

top_quotes_semantic = get_top_quotes_semantic(
    test_query,
    filtered_sentences,
    sentence_embeddings,
    top_k=5
)

for i, q in enumerate(top_quotes_semantic, 1):
    print(f"{i}. score={q['score']:.3f}")
    print(f"   {q['sentence']}\n")

1. score=0.867
   In contrast to RNN sequence-to-sequence models [37], the Transformer outperforms the Berkeley- Parser [29] even when training only on the WSJ training set of 40K sentences.

2. score=0.865
   In this work we propose the Transformer, a model architecture eschewing recurrence and instead relying entirely on an attention mechanism to draw global dependencies between input and output.

3. score=0.856
   For translation tasks, the Transformer can be trained significantly faster than architectures based on recurrent or convolutional layers.

4. score=0.849
   In the Transformer this is reduced to a constant number of operations, albeit at the cost of reduced effective resolution due to averaging attention-weighted positions, an effect we counteract with Multi-Head Attention as described in section 3.

5. score=0.849
   To the best of our knowledge, however, the Transformer is the first transduction model relying entirely on self-attention to compute representations of its i

Summary of Findings

We built a quote-grounded retrieval pipeline to prevent LLM misquoting. Our initial baseline used keyword-based sentence matching, which ensured quotes were valid but often retrieved irrelevant or noisy text (e.g., references, headers).

We then improved retrieval using semantic embeddings, which significantly increased the relevance and quality of extracted sentences. The system now consistently retrieves meaningful, on-topic quotes about the Transformer’s advantages over recurrent models.

However, we observe that while semantic retrieval improves relevance, it does not always rank the most explanatory or causal quotes highest. This highlights that retrieval quality—not quote generation—is the primary bottleneck in building reliable quote-grounded LLM systems.

### what i want to be able to ask is find quotes that support this claim. and LLM will respone somethine like ok here are 3 quotes you may find use ful 1 ) "   " located at ___ 2) ... 3)...  so the tool should also return location

In [20]:
import re

def parse_quote_request(user_query: str):
    q = user_query.lower()

    # detect if user wants quotes
    quote_triggers = [
        "quote", "quotes", "exact quote", "verbatim",
        "support", "evidence", "cite"
    ]

    wants_quotes = any(t in q for t in quote_triggers)

    # extract number of quotes (default = 3)
    match = re.search(r'(\d+)', q)
    if match:
        num_quotes = int(match.group(1))
        num_quotes = min(num_quotes, 10)  # cap
    else:
        num_quotes = 3

    return wants_quotes, num_quotes


# test
print(parse_quote_request("give me 2 quotes supporting this claim"))
print(parse_quote_request("find quotes about transformer"))

(True, 2)
(True, 3)


In [21]:
def format_quotes(quotes):
    lines = []
    for i, q in enumerate(quotes, 1):
        lines.append(
            f'{i}) "{q["sentence"]}"\n'
            f'   Location: start_char={q["start_char"]}, end_char={q["end_char"]}'
        )
    return "\n\n".join(lines)

In [22]:
def quote_tool(user_query, sentences, sentence_embeddings):
    wants_quotes, num_quotes = parse_quote_request(user_query)

    if not wants_quotes:
        return None  # let LLM handle normally

    # semantic retrieval
    quotes = get_top_quotes_semantic(
        user_query,
        sentences,
        sentence_embeddings,
        top_k=num_quotes
    )

    # verify
    for q in quotes:
        q["verified"] = verify_quote_in_source(q["sentence"], article_text)

    # format output
    output = format_quotes(quotes)

    return f"Here are {len(quotes)} quotes you may find useful:\n\n{output}"

In [23]:
def generate_answer(user_query, article_title, article_text, sentences, sentence_embeddings):

    # 1. TRY TOOL FIRST
    tool_output = quote_tool(user_query, sentences, sentence_embeddings)

    if tool_output is not None:
        return {
            "mode": "quote_tool",
            "output": tool_output
        }

    # 2. OTHERWISE USE LLM (normal mode)
    messages = [
        {"role": "system", "content": "You are a helpful assistant."},
        {"role": "user", "content": user_query},
    ]

    text = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True
    )

    inputs = tokenizer(text, return_tensors="pt").to(model.device)

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=150,
            do_sample=False
        )

    decoded = tokenizer.decode(outputs[0], skip_special_tokens=True)

    return {
        "mode": "llm",
        "output": decoded
    }

In [24]:
query = "Give me 3 quotes supporting the claim that the Transformer is better than recurrent models"

result = generate_answer(
    query,
    article_title,
    article_text,
    filtered_sentences,
    sentence_embeddings
)

print(result["mode"])
print(result["output"])

quote_tool
Here are 3 quotes you may find useful:

1) "In contrast to RNN sequence-to-sequence models [37], the Transformer outperforms the Berkeley- Parser [29] even when training only on the WSJ training set of 40K sentences."
   Location: start_char=28744, end_char=28917

2) "In the Transformer this is reduced to a constant number of operations, albeit at the cost of reduced effective resolution due to averaging attention-weighted positions, an effect we counteract with Multi-Head Attention as described in section 3."
   Location: start_char=5369, end_char=5615

3) "In the following sections, we will describe the Transformer, motivate self-attention and discuss its advantages over models such as [17, 18] and [9]."
   Location: start_char=6457, end_char=6607


quote_tool
Here are 3 quotes you may find useful:

1) "In contrast to RNN sequence-to-sequence models [37], the Transformer outperforms the Berkeley- Parser [29] even when training only on the WSJ training set of 40K sentences."
   Location: start_char=28744, end_char=28917

2) "In the Transformer this is reduced to a constant number of operations, albeit at the cost of reduced effective resolution due to averaging attention-weighted positions, an effect we counteract with Multi-Head Attention as described in section 3."
   Location: start_char=5369, end_char=5615

3) "In the following sections, we will describe the Transformer, motivate self-attention and discuss its advantages over models such as [17, 18] and [9]."
   Location: start_char=6457, end_char=6607



### OPTION A (BEST NEXT STEP): Reranking (NO fine-tuning yet)

In [25]:
!pip install sentence-transformers

In [26]:
from sentence_transformers import CrossEncoder

reranker = CrossEncoder("cross-encoder/ms-marco-MiniLM-L-6-v2")

config.json:   0%|          | 0.00/794 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/90.9M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/105 [00:00<?, ?it/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/132 [00:00<?, ?B/s]

In [27]:
def rerank_quotes(query, candidates, top_k=3):
    pairs = [(query, c["sentence"]) for c in candidates]

    scores = reranker.predict(pairs)

    for c, s in zip(candidates, scores):
        c["rerank_score"] = float(s)

    ranked = sorted(candidates, key=lambda x: x["rerank_score"], reverse=True)

    return ranked[:top_k]

In [28]:
# ============================================================
# RE-RUN SEMANTIC + RERANK PIPELINE CONSISTENTLY
# ============================================================

# 1) Make sure filtered_sentences exists
def is_good_sentence(s):
    s_low = s.lower()

    if len(s.strip()) < 40:
        return False
    if "table" in s_low or "figure" in s_low:
        return False
    if "@" in s_low:
        return False
    if "references" in s_low:
        return False
    if "in the following sections" in s_low:
        return False
    if "we will describe" in s_low:
        return False
    if re.match(r'^\d+\s', s_low):   # lines starting with section/page numbers
        return False

    return True

filtered_sentences = [s for s in sentences if is_good_sentence(s["sentence"])]
print("Filtered sentence count:", len(filtered_sentences))


# 2) Rebuild embeddings ONLY on filtered_sentences
sentence_texts = [s["sentence"] for s in filtered_sentences]
sentence_embeddings = embed_text(sentence_texts, prefix="passage")

print("Embeddings shape:", sentence_embeddings.shape)
print("Filtered sentences:", len(filtered_sentences))


# 3) Safer semantic retrieval function
def get_top_quotes_semantic(query, sentence_pool, sentence_embeddings, top_k=10):
    """
    sentence_pool must be the SAME list used to build sentence_embeddings
    """
    if len(sentence_pool) != sentence_embeddings.shape[0]:
        raise ValueError(
            f"Mismatch: len(sentence_pool)={len(sentence_pool)} "
            f"but embeddings rows={sentence_embeddings.shape[0]}"
        )

    query_emb = embed_text([query], prefix="query")  # shape (1, d)
    scores = torch.matmul(sentence_embeddings, query_emb.T).squeeze(-1)

    scored = []
    for i, item in enumerate(sentence_pool):
        scored.append({
            "sentence": item["sentence"],
            "score": float(scores[i].item()),
            "start_char": item["start_char"],
            "end_char": item["end_char"],
        })

    scored.sort(key=lambda x: x["score"], reverse=True)
    return scored[:top_k]




# 5) Load reranker
from sentence_transformers import CrossEncoder

reranker = CrossEncoder("cross-encoder/ms-marco-MiniLM-L-6-v2")
print("Reranker loaded")


# 6) Reranking function
def rerank_quotes(query, candidates, top_k=3):
    pairs = [(query, c["sentence"]) for c in candidates]
    scores = reranker.predict(pairs)

    reranked = []
    for cand, score in zip(candidates, scores):
        cand_copy = cand.copy()
        cand_copy["rerank_score"] = float(score)
        reranked.append(cand_copy)

    reranked.sort(key=lambda x: x["rerank_score"], reverse=True)
    return reranked[:top_k]



# 7) Test upgraded ranking
query = "Why is the Transformer better than recurrent models?"

initial = get_top_quotes_semantic(
    query,
    filtered_sentences,
    sentence_embeddings,
    top_k=10
)

print("INITIAL RETRIEVAL\n")
for i, q in enumerate(initial, 1):
    print(f"{i}. semantic_score={q['score']:.3f}")
    print(f"   {q['sentence']}\n")




# 8) Apply reranker
quotes = rerank_quotes(query, initial, top_k=5)

print("RERANKED RESULTS\n")
for i, q in enumerate(quotes, 1):
    print(f"{i}. rerank_score={q['rerank_score']:.3f} | semantic_score={q['score']:.3f}")
    print(f"   chars=({q['start_char']}, {q['end_char']})")
    print(f"   {q['sentence']}\n")

Filtered sentence count: 250
Embeddings shape: torch.Size([250, 384])
Filtered sentences: 250


Loading weights:   0%|          | 0/105 [00:00<?, ?it/s]

Reranker loaded
INITIAL RETRIEVAL

1. semantic_score=0.867
   In contrast to RNN sequence-to-sequence models [37], the Transformer outperforms the Berkeley- Parser [29] even when training only on the WSJ training set of 40K sentences.

2. semantic_score=0.865
   In this work we propose the Transformer, a model architecture eschewing recurrence and instead relying entirely on an attention mechanism to draw global dependencies between input and output.

3. semantic_score=0.856
   For translation tasks, the Transformer can be trained significantly faster than architectures based on recurrent or convolutional layers.

4. semantic_score=0.849
   In the Transformer this is reduced to a constant number of operations, albeit at the cost of reduced effective resolution due to averaging attention-weighted positions, an effect we counteract with Multi-Head Attention as described in section 3.

5. semantic_score=0.849
   To the best of our knowledge, however, the Transformer is the first transduct

In [29]:
# 9) Verify exact substring
for q in quotes:
    q["verified"] = verify_quote_in_source(q["sentence"], article_text)

print("VERIFIED RERANKED RESULTS\n")
for i, q in enumerate(quotes, 1):
    print(f"{i}. verified={q['verified']}")
    print(f'   "{q["sentence"]}"')
    print(f"   Location: start_char={q['start_char']}, end_char={q['end_char']}\n")

VERIFIED RERANKED RESULTS

1. verified=True
   "For translation tasks, the Transformer can be trained significantly faster than architectures based on recurrent or convolutional layers."
   Location: start_char=29158, end_char=29296

2. verified=True
   "In this work we propose the Transformer, a model architecture eschewing recurrence and instead relying entirely on an attention mechanism to draw global dependencies between input and output."
   Location: start_char=4394, end_char=4586

3. verified=True
   "To the best of our knowledge, however, the Transformer is the first transduction model relying entirely on self-attention to compute representations of its input and output without using sequence- aligned RNNs or convolution."
   Location: start_char=6231, end_char=6457

4. verified=True
   "In terms of computational complexity, self-attention layers are faster than recurrent layers when the sequence 6 length n is smaller than the representation dimensionality d, which is most ofte

In [30]:
# 10) Optional: upgraded quote tool using reranking
def quote_tool_reranked(user_query, filtered_sentences, sentence_embeddings, full_text, top_k_default=3):
    wants, num_quotes = parse_quote_request(user_query)
    if not wants:
        return None

    initial = get_top_quotes_semantic(
        user_query,
        filtered_sentences,
        sentence_embeddings,
        top_k=max(10, num_quotes * 3)
    )

    quotes = rerank_quotes(user_query, initial, top_k=num_quotes)

    for q in quotes:
        q["verified"] = verify_quote_in_source(q["sentence"], full_text)

    lines = []
    for i, q in enumerate(quotes, 1):
        lines.append(
            f'{i}) "{q["sentence"]}"\n'
            f'   Location: start_char={q["start_char"]}, end_char={q["end_char"]}\n'
            f'   Verified: {q["verified"]}'
        )

    return "Here are {} quotes you may find useful:\n\n{}".format(len(quotes), "\n\n".join(lines))

In [31]:
# 11) Final test of upgraded quote tool
test_query = "Give me 3 quotes supporting the claim that the Transformer is better than recurrent models."

tool_output = quote_tool_reranked(
    test_query,
    filtered_sentences,
    sentence_embeddings,
    article_text
)

print(tool_output)

Here are 3 quotes you may find useful:

1) "For translation tasks, the Transformer can be trained significantly faster than architectures based on recurrent or convolutional layers."
   Location: start_char=29158, end_char=29296
   Verified: True

2) "In this work we propose the Transformer, a model architecture eschewing recurrence and instead relying entirely on an attention mechanism to draw global dependencies between input and output."
   Location: start_char=4394, end_char=4586
   Verified: True

3) "To the best of our knowledge, however, the Transformer is the first transduction model relying entirely on self-attention to compute representations of its input and output without using sequence- aligned RNNs or convolution."
   Location: start_char=6231, end_char=6457
   Verified: True


✅ What We’ve Done So Far

We built a quote-grounded retrieval tool that prevents LLMs from hallucinating quotes.

🔹 Core pipeline

1. User query → detect quote intent
    * Checks if user explicitly asks for quotes
    * Extracts how many quotes to return (default = 3)
2. Sentence extraction
    * Split article into sentences
    * Filter out noise (headers, references, short lines, etc.)
3. Semantic retrieval (embedding-based)
    * Uses sentence embeddings to find relevant sentences
    * Much better than initial keyword baseline
4. Reranking (major improvement)
    * Cross-encoder reranker selects sentences that actually support the claim
    * Fixes issue where relevant but useless sentences were returned
5. Verification layer
    * Ensures every quote is an exact substring of the source text
    * Eliminates hallucinated quotes
6. Final quote tool output
    * Returns:
        * exact quotes
        * character locations
        * verified = True

🚧 What’s Next

1. 🔍 Build Evaluation Set (MOST IMPORTANT)

Create ~20–30 test queries:

* quote requests
* non-quote requests
* “support this claim” style queries

Evaluate:

* quote relevance
* correctness (exact match)
* intent detection accuracy

⸻

2. 🧠 Improve Output (small upgrade)

Add:

* short explanation per quote
    (“why this quote supports the claim”)

⸻

3. 📈 Compare Methods (for project)

4. 🧪 THEN consider fine-tuning (optional)

Only if needed for:

* better ranking
* better explanations
* smarter tool usage


5.  Nice UI???

The Evaluation Set
Direct Quote Requests (Q1-Q6): These perfectly test the exact match verification logic and the ability to extract the correct expected_count.

Claim Support (Q7-Q12): This is where the cross-encoder reranker will be truly tested. These queries don't just ask for a specific word, but ask for conceptual support (e.g., Q10 on recurrence not being necessary), requiring the model to align the semantic meaning of the question with the text.

Non-quote (Q13-Q18) & Edge Cases (Q19-Q24): These are vital for ensuring the LLM doesn't get stuck in a "quote extraction loop" and can gracefully handle ambiguous phrasing, ultimately testing the robustness of the wants_quotes classifier.

If you guys have better idea or examples on the evaluation_data set. Feel free to make changes!

In [32]:
import pandas as pd

evaluation_data = [
    {
        "query_id": "Q1",
        "query": "Give me 2 exact quotes about why the Transformer is better than recurrent models.",
        "expected_mode": "quote",
        "expected_behavior": "Return 2 verified relevant quotes",
        "category": "direct_quote"
    },
    {
        "query_id": "Q2",
        "query": "Find 3 direct quotes about training speed in the Transformer paper.",
        "expected_mode": "quote",
        "expected_behavior": "Return quotes about speed/efficiency",
        "category": "direct_quote"
    },
    {
        "query_id": "Q3",
        "query": "Give me 2 verbatim quotes showing that the model relies on attention instead of recurrence.",
        "expected_mode": "quote",
        "expected_behavior": "Return quotes about attention replacing recurrence",
        "category": "direct_quote"
    },
    {
        "query_id": "Q4",
        "query": "Quote the paper’s explanation of why self-attention is useful.",
        "expected_mode": "quote",
        "expected_behavior": "Return relevant conceptual quotes",
        "category": "direct_quote"
    },
    {
        "query_id": "Q5",
        "query": "Give me 3 quotes about the advantages of the Transformer over CNNs or RNNs.",
        "expected_mode": "quote",
        "expected_behavior": "Return comparison-related quotes",
        "category": "direct_quote"
    },
    {
        "query_id": "Q6",
        "query": "Find exact quotes about parallelization in the Transformer.",
        "expected_mode": "quote",
        "expected_behavior": "Return quotes about parallelization",
        "category": "direct_quote"
    },
    {
        "query_id": "Q7",
        "query": "Find 3 quotes supporting the claim that the Transformer trains faster than recurrent models.",
        "expected_mode": "quote",
        "expected_behavior": "Return supporting evidence quotes",
        "category": "claim_support"
    },
    {
        "query_id": "Q8",
        "query": "Find evidence from the paper supporting the claim that self-attention reduces sequential computation.",
        "expected_mode": "quote",
        "expected_behavior": "Return supporting evidence quotes",
        "category": "claim_support"
    },
    {
        "query_id": "Q9",
        "query": "Give quotes supporting the claim that the Transformer achieves strong translation quality.",
        "expected_mode": "quote",
        "expected_behavior": "Return performance-related evidence",
        "category": "claim_support"
    },
    {
        "query_id": "Q10",
        "query": "Find quotes that support the claim that recurrence is not necessary in this model.",
        "expected_mode": "quote",
        "expected_behavior": "Return evidence about no recurrence",
        "category": "claim_support"
    },
    {
        "query_id": "Q11",
        "query": "Cite sentences supporting the claim that the Transformer is more parallelizable.",
        "expected_mode": "quote",
        "expected_behavior": "Return cited supporting sentences",
        "category": "claim_support"
    },
    {
        "query_id": "Q12",
        "query": "What quotes in the paper support the idea that the Transformer is simpler than prior sequence transduction models?",
        "expected_mode": "quote",
        "expected_behavior": "Return supporting quotes if available",
        "category": "claim_support"
    },
    {
        "query_id": "Q13",
        "query": "Summarize why the Transformer was important.",
        "expected_mode": "non_quote",
        "expected_behavior": "Provide normal summary",
        "category": "non_quote"
    },
    {
        "query_id": "Q14",
        "query": "What is the main contribution of the paper?",
        "expected_mode": "non_quote",
        "expected_behavior": "Provide normal answer",
        "category": "non_quote"
    },
    {
        "query_id": "Q15",
        "query": "Explain self-attention in simple terms.",
        "expected_mode": "non_quote",
        "expected_behavior": "Provide explanation, not quotes",
        "category": "non_quote"
    },
    {
        "query_id": "Q16",
        "query": "Why did this paper become influential?",
        "expected_mode": "non_quote",
        "expected_behavior": "Provide high-level answer",
        "category": "non_quote"
    },
    {
        "query_id": "Q17",
        "query": "Compare the Transformer with recurrent models.",
        "expected_mode": "non_quote",
        "expected_behavior": "Provide comparison answer",
        "category": "non_quote"
    },
    {
        "query_id": "Q18",
        "query": "What tasks were used in the experiments?",
        "expected_mode": "non_quote",
        "expected_behavior": "Provide factual answer",
        "category": "non_quote"
    },
    {
        "query_id": "Q19",
        "query": "Can you support your answer with evidence from the paper?",
        "expected_mode": "quote",
        "expected_behavior": "Likely use quote/evidence mode",
        "category": "edge_case"
    },
    {
        "query_id": "Q20",
        "query": "Summarize the paper and cite evidence if needed.",
        "expected_mode": "ambiguous",
        "expected_behavior": "Mixed behavior acceptable; useful for trigger analysis",
        "category": "edge_case"
    },
    {
        "query_id": "Q21",
        "query": "What does the paper say about recurrence?",
        "expected_mode": "non_quote",
        "expected_behavior": "Should likely remain normal QA",
        "category": "edge_case"
    },
    {
        "query_id": "Q22",
        "query": "Give me one quote and then explain it.",
        "expected_mode": "quote",
        "expected_behavior": "Return quote plus explanation",
        "category": "edge_case"
    },
    {
        "query_id": "Q23",
        "query": "Is the Transformer better than RNNs?",
        "expected_mode": "non_quote",
        "expected_behavior": "Answer normally without quote tool",
        "category": "edge_case"
    },
    {
        "query_id": "Q24",
        "query": "Find 3 quotes supporting the claim that the Transformer solves every NLP problem better than all previous models.",
        "expected_mode": "quote",
        "expected_behavior": "Should fail gracefully and avoid overstated support",
        "category": "edge_case"
    }
]

eval_df = pd.DataFrame(evaluation_data)

# show the dataframe
eval_df

,query_id,query,expected_mode,expected_behavior,category
0,Q1,Give me 2 exact quotes about why the Transform...,quote,Return 2 verified relevant quotes,direct_quote
1,Q2,Find 3 direct quotes about training speed in t...,quote,Return quotes about speed/efficiency,direct_quote
2,Q3,Give me 2 verbatim quotes showing that the mod...,quote,Return quotes about attention replacing recurr...,direct_quote
3,Q4,Quote the paper’s explanation of why self-atte...,quote,Return relevant conceptual quotes,direct_quote
4,Q5,Give me 3 quotes about the advantages of the T...,quote,Return comparison-related quotes,direct_quote
5,Q6,Find exact quotes about parallelization in the...,quote,Return quotes about parallelization,direct_quote
6,Q7,Find 3 quotes supporting the claim that the Tr...,quote,Return supporting evidence quotes,claim_support
7,Q8,Find evidence from the paper supporting the cl...,quote,Return supporting evidence quotes,claim_support
8,Q9,Give quotes supporting the claim that the Tran...,quote,Return performance-related evidence,claim_support
9,Q10,Find quotes that support the claim that recurr...,quote,Return evidence about no recurrence,claim_support


In [33]:
import pandas as pd
import numpy as np
import os
import wandb
from scipy import stats

def get_wandb_api_key():
    """Return a W&B API key from Colab Secrets or the local environment, if available."""
    try:
        from google.colab import userdata
        key = userdata.get("WANDB_API_KEY")
        if key:
            return key
    except Exception:
        pass
    return os.getenv("WANDB_API_KEY")

def evaluate_pipeline(eval_df, source_text, pipeline_func, llm_judge_func, run_name="pipeline_eval", use_wandb=True):
    """
    Evaluates the pipeline using a Pandas DataFrame directly.
    If a WANDB_API_KEY is available, optionally logs results to Weights & Biases.
    """
    # 1. Initialize optional Weights & Biases tracking
    wandb_run = None
    if use_wandb:
        wandb_api_key = get_wandb_api_key()
        if wandb_api_key:
            wandb.login(key=wandb_api_key)
            wandb_run = wandb.init(project="5293-retrieval-project", name=run_name)
        else:
            print("W&B logging skipped. Set WANDB_API_KEY as a Colab secret or environment variable to enable it.")

    # Note: pd.read_csv() was removed. We now iterate directly over the passed eval_df.
    results = []

    for index, row in eval_df.iterrows():
        query_id = row['query_id']
        query = row['query']
        expected_intent = row['expected_mode']

        # 2. Run the pipeline
        # Assuming the pipeline returns a dictionary with intent, quotes, and the raw retrieved contexts
        output = pipeline_func(query, source_text)
        predicted_intent = output['intent']
        retrieved_quotes = output['quotes']
        top_k_contexts = output['raw_retrieved_contexts']

        # --- SCORING RUBRIC ---

        # Metric A: Intent Correct (1 or 0)
        intent_correct = 1 if predicted_intent == expected_intent else 0

        # Metric B: Stayed out of quote mode (for non-quote queries)
        stayed_out_of_quote_mode = 1 if expected_intent == "non_quote" and predicted_intent == "non_quote" else 0

        # Metric C: Correct number of quotes (1 or 0)
        expected_count = output.get('requested_count', 0)
        correct_number = 1 if len(retrieved_quotes) == expected_count and expected_intent == "quote" else 0

        # Metric D: Exact Source Quotes (1 or 0)
        exact_match = 0
        if len(retrieved_quotes) > 0:
            matches = sum([1 for q in retrieved_quotes if q in source_text])
            exact_match = 1 if matches == len(retrieved_quotes) else 0

        # Metric E: Retrieval vs Generation Split (Top-K Context Check)
        relevant_in_top_k = 0
        if len(top_k_contexts) > 0:
            retrieval_prompt = f"Query: {query}\nContexts: {top_k_contexts}\nIs the answer present in these contexts? (YES/NO)"
            if "YES" in llm_judge_func(retrieval_prompt).upper():
                relevant_in_top_k = 1

        # Metric F & G: Answer Quality & Quote Relevance (0, 1, or 2)
        relevance_score = 0
        raw_judge_output = ""

        if predicted_intent == "non_quote":
            qa_prompt = f"Query: {query}\nAnswer: {output['generation']}\nRate the answer quality strictly as 0 (poor), 1 (acceptable), or 2 (good)."
            raw_judge_output = llm_judge_func(qa_prompt)
        elif len(retrieved_quotes) > 0:
            quote_prompt = f"Query: {query}\nQuotes: {retrieved_quotes}\nRate how relevant these quotes are to the query strictly as 0 (not relevant), 1 (partly relevant), or 2 (highly relevant)."
            raw_judge_output = llm_judge_func(quote_prompt)

        # Safely extract the first digit found in the Qwen response
        match = re.search(r'\d', raw_judge_output)
        if match:
            relevance_score = int(match.group(0))
        else:
            relevance_score = 0 # Fallback if Qwen returns letters only

        # Store row results
        results.append({
            "query_id": query_id,
            "query": query,
            "expected_category": row['category'],
            "intent_correct": intent_correct,
            "stayed_out_of_quote_mode": stayed_out_of_quote_mode,
            "exact_source_quotes": exact_match,
            "correct_number_of_quotes": correct_number,
            "relevant_in_top_k": relevant_in_top_k,
            "relevance_quality_score": relevance_score
        })

    results_df = pd.DataFrame(results)

    # 3. Log the final table and aggregate metrics to Weights & Biases when enabled
    if wandb_run:
        wandb_table = wandb.Table(dataframe=results_df)
        wandb.log({f"{run_name}_evaluation_results": wandb_table})
        wandb.log({
            "avg_intent_accuracy": results_df['intent_correct'].mean(),
            "avg_exact_match": results_df['exact_source_quotes'].mean(),
            "avg_relevance_score": results_df['relevance_quality_score'].mean(),
            "retrieval_success_rate": results_df['relevant_in_top_k'].mean()
        })
        wandb.finish()
    return results_df

# To run it, you now just pass your eval_df directly:
# final_results = evaluate_pipeline(eval_df, source_text, your_pipeline, your_judge_llm)

In [34]:
def compare_pipeline_significance(baseline_df, semantic_df):
    """
    Runs a Kruskal-Wallis H-test to determine if the semantic pipeline's
    relevance scores are statistically significantly different from the baseline.
    """
    baseline_scores = baseline_df['relevance_quality_score'].values
    semantic_scores = semantic_df['relevance_quality_score'].values

    # Perform Kruskal-Wallis Test
    stat, p_value = stats.kruskal(baseline_scores, semantic_scores)

    print("=== Statistical Comparison (Kruskal-Wallis) ===")
    print(f"H-Statistic: {stat:.4f}")
    print(f"P-Value:     {p_value:.4f}")

    alpha = 0.05
    if p_value < alpha:
        print("Result: Statistically Significant. The semantic method performs differently than the baseline.")
        # Calculate mean to show the direction of improvement
        if semantic_scores.mean() > baseline_scores.mean():
            print("Conclusion: The semantic/cross-encoder pipeline is a statistically proven improvement.")
    else:
        print("Result: Not Significant. We fail to reject the null hypothesis.")

# Example execution:
# baseline_results = evaluate_pipeline("eval_set.csv", article, naive_baseline_func, qwen_judge, "baseline_run")
# semantic_results = evaluate_pipeline("eval_set.csv", article, semantic_pipeline_func, qwen_judge, "semantic_run")
# compare_pipeline_significance(baseline_results, semantic_results)

In [35]:
# Install a PDF reader library if you don't have it
!pip install PyPDF2 requests

import requests
import PyPDF2
import io

# URL for the "Attention Is All You Need" paper on arXiv
arxiv_url = "https://arxiv.org/pdf/1706.03762.pdf"

print("Downloading paper...")
response = requests.get(arxiv_url)

# Read the PDF directly from the downloaded bytes
pdf_file = io.BytesIO(response.content)
reader = PyPDF2.PdfReader(pdf_file)

# Extract all text into your source_text variable
source_text = ""
for page in reader.pages:
    extracted = page.extract_text()
    if extracted:
        source_text += extracted + "\n"

print(f"Paper loaded successfully! Total length: {len(source_text)} characters.")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 232.6/232.6 kB 22.5 MB/s eta 0:00:00
Paper loaded successfully! Total length: 39487 characters.


In [36]:
def my_pipeline_wrapper(query, source_text):
    # STEP 1: Intent Parsing
    wants_quotes, requested_count = parse_quote_request(query)
    predicted_intent = "quote" if wants_quotes else "non_quote"

    # STEP 2: Semantic Retrieval
    # Grab a large initial pool (top 10) from the pre-computed embeddings
    initial_candidates = get_top_quotes_semantic(
        query=query,
        sentence_pool=filtered_sentences,
        sentence_embeddings=sentence_embeddings,
        top_k=10
    )

    # STEP 3: Reranking
    # Cross-encode the top 10, and keep only the requested number of quotes
    # (e.g., if the user asked for 3, keep the top 3 reranked)
    reranked_dicts = rerank_quotes(
        query=query,
        candidates=initial_candidates,
        top_k=requested_count
    )

    # Extract just the text sentences from the dictionaries
    top_contexts = [item["sentence"] for item in reranked_dicts]

    # STEP 4: Final Output Generation
    # Since your snippet doesn't include the final Qwen generation call,
    # we route the logic here:
    if predicted_intent == "quote":
        # If they want quotes, we just return the exact retrieved strings
        final_quotes = top_contexts
        final_answer = ""
    else:
        # If it's a non_quote query, no quotes are returned,
        # and you would call Qwen here to generate a summary
        final_quotes = []

        # NOTE: Replace this string with your actual Qwen generation function
        final_answer = "This is where the Qwen-generated summary goes."

    # Return the standardized dictionary for the evaluator
    return {
        "intent": predicted_intent,
        "requested_count": requested_count,
        "quotes": final_quotes,
        "raw_retrieved_contexts": top_contexts,
        "generation": final_answer
    }

In [37]:
def qwen_judge(eval_prompt):
    """
    Takes a grading prompt, formats it using the groupmate's chat template,
    and returns ONLY the model's generated text response.
    """
    messages = [
        {"role": "system", "content": "You are a strict, objective evaluator. Follow the prompt instructions exactly and output only the requested score or YES/NO."},
        {"role": "user", "content": eval_prompt}
    ]

    # Use the same chat template application as Cell 7
    text = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True
    )

    # Move inputs to the device mapping already set up by your groupmate
    inputs = tokenizer(text, return_tensors="pt").to(model.device)

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=10,
            temperature=0.1 # Low temperature so the judge is deterministic
        )

    # SLICE the output to remove the prompt tokens and keep only the new answer
    input_length = inputs.input_ids.shape[1]
    generated_ids = outputs[0][input_length:]

    # Decode only the generated answer
    response = tokenizer.decode(generated_ids, skip_special_tokens=True)

    return response.strip()

# Quick test to make sure it only returns "YES" and not the whole prompt
# print(qwen_judge("Is the sky blue? Answer YES/NO."))

In [38]:
# Optional: set WANDB_API_KEY as a Colab secret or environment variable to log this run to W&B.
# Without a key, evaluate_pipeline still returns the local results dataframe.

# Run the evaluation.
final_results_df = evaluate_pipeline(
    eval_df=eval_df,
    source_text=source_text,
    pipeline_func=my_pipeline_wrapper,
    llm_judge_func=qwen_judge,
    run_name="baseline_evaluation"
)

# Display the final dataframe in Colab
display(final_results_df)

/usr/local/lib/python3.12/dist-packages/notebook/notebookapp.py:191: SyntaxWarning: invalid escape sequence '\/'
  | |_| | '_ \/ _` / _` |  _/ -_)
wandb: WARNING If you're specifying your api key in code, ensure this code is not shared publicly.
wandb: WARNING Consider setting the WANDB_API_KEY environment variable, or running `wandb login` from the command line.
wandb: [wandb.login()] Using explicit session credentials for https://api.wandb.ai.
wandb: No netrc file found, creating one.
wandb: Appending key for api.wandb.ai to your netrc file: /root/.netrc
wandb: Currently logged in as: xz3457 (xz3457-columbia-university) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


avg_exact_match,▁
avg_intent_accuracy,▁
avg_relevance_score,▁
retrieval_success_rate,▁
avg_exact_match,0
avg_intent_accuracy,0.95833
avg_relevance_score,0.70833
retrieval_success_rate,0.29167


,query_id,query,expected_category,intent_correct,stayed_out_of_quote_mode,exact_source_quotes,correct_number_of_quotes,relevant_in_top_k,relevance_quality_score
0,Q1,Give me 2 exact quotes about why the Transform...,direct_quote,1,0,0,1,1,2
1,Q2,Find 3 direct quotes about training speed in t...,direct_quote,1,0,0,1,0,1
2,Q3,Give me 2 verbatim quotes showing that the mod...,direct_quote,1,0,0,1,0,1
3,Q4,Quote the paper’s explanation of why self-atte...,direct_quote,1,0,0,1,0,0
4,Q5,Give me 3 quotes about the advantages of the T...,direct_quote,1,0,0,1,1,2
5,Q6,Find exact quotes about parallelization in the...,direct_quote,1,0,0,1,0,1
6,Q7,Find 3 quotes supporting the claim that the Tr...,claim_support,1,0,0,1,1,2
7,Q8,Find evidence from the paper supporting the cl...,claim_support,1,0,0,1,0,1
8,Q9,Give quotes supporting the claim that the Tran...,claim_support,1,0,0,1,0,1
9,Q10,Find quotes that support the claim that recurr...,claim_support,1,0,0,1,0,0


For the scoring rubric explaination:

1. intent_correct:

1: The system correctly identified whether the query needed a quote or a normal answer.

0: The system guessed the wrong mode.

2. exact_source_quotes:

1: Every single quote returned to the user is a 100% exact, verbatim substring of the source document.

0: The model hallucinated, paraphrased, or altered the quote (even slightly).

3. relevant_in_top_k:

1: The correct information to answer the user's question was successfully retrieved by the e5-small-v2 and ms-marco models and was present in the chunks passed to the LLM.

0: The retrieval pipeline completely missed the right sentences, meaning the LLM was flying blind.

4. relevance_quality_score:

2: Highly relevant / Good answer.

1: Partly relevant / Acceptable answer.

0: Not relevant / Poor answer.

5. stayed_out_of_quote_mode:

1: The system correctly recognized it was a normal question and provided a standard generated answer without forcing any quotes.

0: The system got confused and inappropriately triggered the quote retrieval tool when the user just wanted a normal chat.

6. correct_number_of_quotes:

1: The system returned the exact number of quotes the user asked for.

0: The system returned too many, too few, or none at all.